### Before running this notebook, create a virtual enviornment as described in the [BlueRecording repo](https://github.com/openbraininstitute/BlueRecording?tab=readme-ov-file#bluerecording-1). Note that the spack environment described in that section is not required

In [12]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
from glob import glob
from tqdm import tqdm
import xarray as xr
import os, sys
import scipy
import pandas as pd
from scipy.ndimage import gaussian_filter
from sklearn.decomposition import PCA
from scipy import stats

from scipy import interpolate

from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

from cinplaAnalysis.utils import *

import quantities as pq

%matplotlib ipympl

In [13]:
from cinplaAnalysis.icsd import StandardCSD, DeltaiCSD, StepiCSD

In [14]:
###
# Loads data
###
highRes = np.load('lfp.npy') # Loads LFP and oCSD averaged over trials, 20 um spacing, rho=500um
lfpIdx = np.arange(101) 
totalLFPHighRes = highRes[:,lfpIdx] # LFP, 20 um spacing
totalLFPLowResQuarter = totalLFPHighRes[:,0:totalLFPHighRes.shape[1]:8] # LFP, 160 um spacing

In [15]:
###
# Filters data
###

from scipy.signal import *
sos = butter(10, [1,499], 'bp', fs=1000, output='sos')
totalLFPLowResQuarter = sosfilt(sos, totalLFPLowResQuarter,axis=0)

### First, we calculate the magnitude of the LFP

In [16]:
max_LFP_magnitude = np.max(np.max(totalLFPLowResQuarter[2000:2050],axis=0) - np.min(totalLFPLowResQuarter[2000:2050],axis=0))
print(max_LFP_magnitude)

0.00017780077772605403


### Our peak LFP magnitude is 0.18 mV. In contrast, the [Quairiaux paper](https://doi.org/10.1523/JNEUROSCI.5995-10.2011), the [Reyes-Puerta paper](https://doi.org/10.1093/cercor/bhu007) and [this paper by Einevoll et al.](https://doi.org/10.1152/jn.00845.2006) have all have peak LFP magnitudes of ~1mV. [This paper by Riera et al.](https://doi.org/10.1152/jn.00098.2011) has peak LFP magntitudes of ~0.3 mV.

### Now, we calculate the CSD

In [110]:
### Calculates iCSD from LFP data, under the assumption that the LFP comes from a single column

diam = 500E-6 * pq.m                              # [m]
sigma = 0.376 * pq.S / pq.m                         # [S/m] or [1/(ohm*m)]
sigma_top = 0.376 * pq.S / pq.m                     # [S/m] or [1/(ohm*m)]

dz = 160e-6*pq.m # Inter-electrode distance of 160 um
electrode_positions = np.arange(0,totalLFPLowResQuarter.shape[1]) * dz

step_input_low_quarter = {
    'lfp' : totalLFPLowResQuarter.T  * pq.V,      # [mV] -> [V],
    'coord_electrode' : electrode_positions,
    'sigma' : sigma,
    'h':dz,
    'f_type' : 'gaussian',
    'f_order' : (3, 1),
    'diam': diam
}

icsd_low_quarter = StepiCSD(**step_input_low_quarter)

truecsd = icsd_low_quarter.get_csd()[:,2000:2050]

### Integrating the CSD over distance, we get the dipole density per unit cross-sectional area

In [111]:
# Estimates the dipole density from the CSD, under the assumption that LFP comes from a single column

peak_csd_timepoint = np.where(np.abs(truecsd)==np.max(np.abs(truecsd)))[1][0]

dipoleDensity = np.sum( electrode_positions *truecsd[:,peak_csd_timepoint]*dz)

print(dipoleDensity.rescale(pq.nA*pq.m/pq.mm**2))

-0.2657851672245636 m*nA/mm**2


### We obtain a dipole density of 0.25 $nA*m/mm^2$. In contrast, [this paper](https://doi.org/10.1016/j.neuroimage.2015.02.003) finds a dipole density of 2 $nA*m/mm^2$, using the data from the Riera et al. paper mentioned above.

### Next, we calculate the EEG signals we would expect to obtain, given the dipole density calculated from our CSD, and the E-fields in our finite element model

In [112]:
dipole_from_column = dipoleDensity * np.pi*(250*pq.um)**2

In [113]:
size_of_l5pc = 1*pq.mm
max_efield = 4*pq.V/pq.A/size_of_l5pc

In [114]:
eeg_from_central_column = dipole_from_column * max_efield

In [115]:
print(eeg_from_central_column.rescale(pq.V))

-2.0874718219645588e-07 V


### This calculation produces an EEG signal estimate for a single column that is a factor 2 greater than the EEG BlueRecording produces for the entire O1. However, this can be attributed to the assumption that the LFP is generated entirely by a single column. If in the LFP calculation we set diam to 1000 um, which seems more realistic given the CSD in the Reyes-Puerta paper, then we get an EEG of 2e-8 V from the central column; this is in line with what we expect given a 1e-7 V signal from O1. 